[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C30_Agent_Harness_Course/05_minimal_agent/05_minimal_agent.ipynb)

# 05 · 最小可用 Agent（拼装四块零件，端到端多步任务）

目标：把前四模块的零件——**循环(01) + 工具系统(02) + LLM 适配器(03) + 鲁棒层(04)**——拼成一个完整、可配置、可扩展的 `Agent`，端到端跑通一个**多步研究式任务**、验证它能**自我恢复**、并亲手做一个**扩展**。

路线：复现四块零件(精简) → 组装 Agent 类 → 跑多步研究任务 → 出错自我恢复 → 可配置(换工具/上限) → 扩展(加新工具) → ✏️ 练习 → 📖 答案 → 🧪 接真实 Claude 端到端胶囊。

> 心智模型：**鲁棒层(外) → 循环(中) → {适配器, 工具系统}(被调用)；全靠依赖注入拼起来**。纯标准库可跑、无需 key。

## 1 · 复现四块零件(精简版)

先把前四模块的核心零件精简地放到一起：MockLLM/适配工厂(03)、工具注册表+分发(02)、循环用的状态、鲁棒件(04)。这是组装的原料。

In [ ]:
import os, json, random

# ---- 03: LLM 接口 + MockLLM + 无 key 回退 ----
class MockLLM:
    def __init__(self, script):
        self.script = list(script); self.calls = 0
    def complete(self, system, messages, tools):
        d = dict(self.script[self.calls]); self.calls += 1
        d.setdefault('text',''); d.setdefault('tool_calls',[])
        d.setdefault('usage',{'input_tokens':1000,'output_tokens':500})
        return d

# ---- 02: 工具注册表 + 分发(错误隔离) ----
class ToolRegistry:
    def __init__(self): self._t = {}
    def add(self, name, fn, schema, desc=''):
        self._t[name] = ({'name':name,'description':desc or (fn.__doc__ or '').strip(),
                          'input_schema':schema}, fn)
        return self
    def export(self): return [s for s,_ in self._t.values()]
    def names(self): return list(self._t)
    def __contains__(self, n): return n in self._t
    def __getitem__(self, n): return self._t[n]

def dispatch(reg, call):
    name, args = call['name'], call.get('input', {})
    if name not in reg:
        return {'tool_use_id':call['id'],'content':f'无此工具 {name}，可用: {reg.names()}','is_error':True}
    schema, fn = reg[name]
    for r in schema['input_schema'].get('required', []):
        if r not in args:
            return {'tool_use_id':call['id'],'content':f'缺少必填参数 {r}','is_error':True}
    try:
        return {'tool_use_id':call['id'],'content':str(fn(**args)),'is_error':False}
    except Exception as e:
        return {'tool_use_id':call['id'],'content':f'{type(e).__name__}: {e}','is_error':True}

# ---- 04: 鲁棒件 ----
def estimate_cost(u, model='claude-opus-4-8'):
    pin, pout = {'claude-opus-4-8':(5.0,25.0),'claude-sonnet-4-6':(3.0,15.0),
                 'claude-haiku-4-5':(1.0,5.0)}.get(model,(5.0,25.0))
    return u['input_tokens']/1e6*pin + u['output_tokens']/1e6*pout
class LoopDetector:
    def __init__(self, window=4, threshold=3): self.w, self.th, self.r = window, threshold, []
    def check(self, call):
        fp = (call['name'], json.dumps(call['input'], sort_keys=True, ensure_ascii=False))
        self.r.append(fp); self.r = self.r[-self.w:]
        return self.r.count(fp) >= self.th

print('✅ 四块零件就绪：MockLLM(03) / ToolRegistry+dispatch(02) / cost+LoopDetector(04)')

## 2 · 组装 Agent 类：四块零件严丝合缝

把零件拼成一个 `Agent`：构造时**注入** llm、registry、上限(依赖注入)；`run(task)` 跑完整多步过程、返回带状态与 trace 的结果。
拓扑：**鲁棒层(外) → 循环(中) → {适配器, 工具}(被调用)**。

In [ ]:
class Agent:
    def __init__(self, llm, registry, system='You are a helpful agent.',
                 max_steps=15, max_cost=0.5, model='claude-opus-4-8', loop_threshold=3):
        self.llm, self.registry, self.system = llm, registry, system     # 03 + 02
        self.max_steps, self.max_cost, self.model = max_steps, max_cost, model  # 04
        self.loop_threshold = loop_threshold
    def run(self, task):
        history = [{'role':'user','content':task}]
        total_cost = 0.0; det = LoopDetector(threshold=self.loop_threshold); trace = []
        for step in range(self.max_steps):                       # 01 循环 + 04 步数保险丝
            d = self.llm.complete(self.system, history, self.registry.export())  # 02+03
            total_cost += estimate_cost(d['usage'], self.model)   # 04 成本追踪
            for call in d.get('tool_calls', []):
                trace.append(f'[{step}] ACTION {call["name"]}({call["input"]})')
            if d['text']: trace.append(f'[{step}] THOUGHT/TEXT: {d["text"]}')
            if total_cost > self.max_cost:                        # 04 预算保险丝
                return {'status':'budget_exceeded','trace':trace,'cost':total_cost}
            history.append({'role':'assistant','text':d['text'],'tool_calls':d['tool_calls']})  # 01
            if d['stop_reason'] == 'end_turn':                    # 01 正常停止
                return {'status':'done','answer':d['text'],'trace':trace,'cost':total_cost,'steps':step+1}
            for call in d['tool_calls']:
                if det.check(call):                              # 04 循环检测
                    return {'status':'loop_detected','trace':trace,'cost':total_cost}
            results = [dispatch(self.registry, c) for c in d['tool_calls']]  # 02 分发+隔离
            for r in results:
                trace.append(f'      -> {"ERR " if r["is_error"] else ""}{r["content"]}')
            history.append({'role':'user','content':results})    # 01 观察回灌
        return {'status':'max_steps','trace':trace,'cost':total_cost}

# 烟雾测试：最简单的『直接作答』
reg = ToolRegistry()
a = Agent(MockLLM([{'stop_reason':'end_turn','text':'你好!'}]), reg)
out = a.run('打个招呼')
assert out['status'] == 'done' and out['answer'] == '你好!'
print('✅ Agent 组装完成：依赖注入 llm/registry/上限, run() 串起四块零件')

## 3 · 端到端多步研究式任务：自主串起两个工具

让 agent 解一个**需要多步、有依赖**的任务：『某圆半径 = "agent" 的字母数, 求面积』。它必须先 count_letters 拿到 5, 再用 calculator 算 π·5²。逐步断言整条轨迹。

In [ ]:
reg = ToolRegistry()
reg.add('count_letters', lambda word: len(word),
        {'type':'object','properties':{'word':{'type':'string'}},'required':['word']},
        '数一个单词有几个字母')
reg.add('calculator', lambda expr: round(eval(expr, {'__builtins__':{}}, {}), 2),
        {'type':'object','properties':{'expr':{'type':'string'}},'required':['expr']},
        '计算一个算术表达式')

# MockLLM 脚本 = 真实模型会自主走出的轨迹(这里写死以便逐步断言)
brain = MockLLM([
    {'stop_reason':'tool_use','text':'先数 agent 的字母',
     'tool_calls':[{'id':'1','name':'count_letters','input':{'word':'agent'}}]},
    {'stop_reason':'tool_use','text':'半径5, 算 π·r²',
     'tool_calls':[{'id':'2','name':'calculator','input':{'expr':'3.14159*5*5'}}]},
    {'stop_reason':'end_turn','text':'该圆面积约为 78.54'},
])
agent = Agent(brain, reg, system='你是严谨的研究助手。')
out = agent.run('某圆半径等于 "agent" 这个词的字母数, 求它的面积')
print('\n'.join(out['trace']))
print('\n最终答案:', out['answer'], '| 步数:', out['steps'])
# 逐步断言整条轨迹(MockLLM 确定性的价值)
tr = '\n'.join(out['trace'])
assert out['status'] == 'done' and out['steps'] == 3
assert 'count_letters' in tr        # 第0步数字母
assert '-> 5' in tr                 # 字母数=5 被观察到(回灌)
assert 'calculator' in tr           # 据结果(半径5)算面积
assert tr.index('count_letters') < tr.index('calculator')  # 顺序: 先数字母再算面积
assert '78.54' in out['answer']
print('\n✅ 端到端多步任务跑通：自主串起 count_letters->calculator, 中间结果正确回灌')

## 4 · 逆境检验：出错后自我恢复

顺境能跑还不够。让 agent 中途**调错工具名**(幻觉)，看到错误后**改对**——验证你拼装的 agent 逆境也扛得住(模块02错误隔离 + 模型据错改正)。

In [ ]:
brain2 = MockLLM([
    # 第0步: 调一个不存在的工具(幻觉)
    {'stop_reason':'tool_use','text':'我试试 calc',
     'tool_calls':[{'id':'1','name':'calc','input':{'expr':'2+2'}}]},  # 错! 应叫 calculator
    # 看到错误后改对
    {'stop_reason':'tool_use','text':'哦是 calculator',
     'tool_calls':[{'id':'2','name':'calculator','input':{'expr':'2+2'}}]},
    {'stop_reason':'end_turn','text':'2+2=4'},
])
out = Agent(brain2, reg).run('算 2+2')
print('\n'.join(out['trace']))
assert out['status'] == 'done'
# 第0步应是幻觉工具错误(ERR), agent 没崩
assert any('ERR' in line and '无此工具' in line for line in out['trace']), '幻觉工具应被隔离成 ERR'
# 改对后成功算出 4
assert '4' in out['answer']
print('\n✅ 逆境检验通过：幻觉工具被隔离成可反馈错误 -> agent 据错改正 -> 完成任务(没崩)')

## 5 · 可配置：同一个 Agent，多种用法

依赖注入的回报：换工具集、调上限、换人格都不改 Agent 代码。验证三种配置都各自正确工作。

In [ ]:
# 配置A: 严格止损(小预算) -> 贵任务触发熔断
costly = [{'stop_reason':'tool_use','tool_calls':[{'id':str(i),'name':'calculator','input':{'expr':f'{i}+1'}}],
           'usage':{'input_tokens':500000,'output_tokens':200000}} for i in range(20)]
strict = Agent(MockLLM(costly), reg, max_cost=3.0, max_steps=20)
rA = strict.run('反复算')
assert rA['status'] == 'budget_exceeded', '小预算应熔断'

# 配置B: 换一套工具(只有 echo) -> 同一 Agent, 不同手脚
reg2 = ToolRegistry()
reg2.add('echo', lambda text: text.upper(),
         {'type':'object','properties':{'text':{'type':'string'}},'required':['text']}, '大写回显')
rB = Agent(MockLLM([{'stop_reason':'tool_use','tool_calls':[{'id':'1','name':'echo','input':{'text':'hi'}}]},
                    {'stop_reason':'end_turn','text':'回显完成'}]), reg2).run('回显 hi')
assert rB['status'] == 'done' and any('HI' in l for l in rB['trace'])

# 配置C: 换人格(system) + 换模型(影响成本单价)
rC = Agent(MockLLM([{'stop_reason':'end_turn','text':'简洁回答'}]),
           reg, system='你是极简助手, 只说要点。', model='claude-haiku-4-5').run('问候')
assert rC['status'] == 'done'
print('A(熔断):', rA['status'], '| B(换工具):', rB['status'], '| C(换人格/模型):', rC['status'])
print('✅ 同一个 Agent 类经配置变出三种用法, 核心 run() 一字未改 —— 依赖注入的回报')

## 6 · 扩展：在预留接缝处加新能力

最小 agent 是种子, 不是天花板。体会『在 registry 这个接缝处加新工具』有多平滑——不碰循环、不碰鲁棒层, 只往 registry 加一个工具, agent 立刻会用了。

In [ ]:
# 给已有 agent 加一个全新工具: reverse(把字符串倒过来)
reg.add('reverse', lambda s: s[::-1],
        {'type':'object','properties':{'s':{'type':'string'}},'required':['s']}, '反转字符串')
assert 'reverse' in reg.names(), '新工具应已注册'
# 新工具自动出现在给模型的清单里(不变式: export == 可分发)
assert any(t['name'] == 'reverse' for t in reg.export())

# agent 立刻能用它 —— 循环/鲁棒层一行没改
brain3 = MockLLM([
    {'stop_reason':'tool_use','tool_calls':[{'id':'1','name':'reverse','input':{'s':'agent'}}]},
    {'stop_reason':'end_turn','text':'倒过来是 tnega'},
])
out = Agent(brain3, reg).run('把 agent 倒过来')
assert out['status'] == 'done'
assert any('tnega' in line for line in out['trace']), '新工具应被正确调用'
print('轨迹片段:', [l for l in out['trace'] if 'reverse' in l or 'tnega' in l])
print('✅ 扩展平滑: 只往 registry 加一个工具, agent 立刻会用 —— 好分解让加能力变容易')

---
## ✏️ 练习 1：给 Agent 加『终止原因摘要』

实现 `summarize_run(result)`：把 `Agent.run` 的返回结果总结成一行人类可读的话，覆盖每种 status：
`done`->`'✓ 完成({steps}步, ${cost}): {answer}'`；`budget_exceeded`->`'✗ 超预算(${cost})'`；`loop_detected`->`'✗ 检测到死循环'`；`max_steps`->`'✗ 超过最大步数'`。

In [ ]:
def summarize_run(result):
    # TODO: 按 result['status'] 返回对应的一行摘要(用上 steps/cost/answer 等字段)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert '✓ 完成' in summarize_run({'status':'done','steps':3,'cost':0.01,'answer':'42'})
assert '42' in summarize_run({'status':'done','steps':3,'cost':0.01,'answer':'42'})
assert '超预算' in summarize_run({'status':'budget_exceeded','cost':0.6})
assert '死循环' in summarize_run({'status':'loop_detected','cost':0.1})
assert '最大步数' in summarize_run({'status':'max_steps','cost':0.2})
print('示例:', summarize_run({'status':'done','steps':3,'cost':0.0123,'answer':'面积78.54'}))
print('✅ 练习 1 通过：每种终止状态都有清晰摘要')

## ✏️ 练习 2：构造一个会触发循环检测的 agent

不写循环检测代码(已有)，而是**构造一个会死循环的场景**并验证 agent 拦得住。
写 `make_looping_agent(reg)`：返回一个 `Agent`, 其 MockLLM 脚本让它**反复调用完全相同的 `(工具,参数)`**(如总调 calculator(expr='1+1'))，使 `run` 返回 `status=='loop_detected'`(在 max_steps 之前)。

In [ ]:
def make_looping_agent(reg, loop_threshold=3):
    # TODO: 构造一个 MockLLM 脚本: 重复很多次相同的 tool_use(calculator, expr='1+1')
    #       返回 Agent(MockLLM(脚本), reg, max_steps=20, loop_threshold=loop_threshold)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
ag = make_looping_agent(reg, loop_threshold=3)
out = ag.run('反复算 1+1')
assert out['status'] == 'loop_detected', '反复同一动作应触发循环检测'
# 应在 max_steps(20) 之前就被拦下
assert len([l for l in out['trace'] if 'ACTION' in l]) < 20
print(f'循环检测在第 {len([l for l in out["trace"] if "ACTION" in l])} 个动作处止损')
print('✅ 练习 2 通过：能造出死循环场景并验证 agent 精准止损(早于 max_steps)')

## ✏️ 练习 3：agent vs workflow 的判断

Anthropic《Building effective agents》：不是所有任务都该用 agent。实现 `should_use_agent(task)` 的简化启发式：
若任务描述含『固定/每次都/按顺序/批量』等**步骤可预先确定**的信号词 -> 返回 `'workflow'`；
若含『研究/探索/根据情况/不确定』等**开放式**信号词 -> 返回 `'agent'`；都没有 -> 返回 `'workflow'`(默认更可控)。

In [ ]:
def should_use_agent(task):
    WORKFLOW_HINTS = ['固定', '每次都', '按顺序', '批量', '总是']
    AGENT_HINTS = ['研究', '探索', '根据情况', '不确定', '自己决定']
    # TODO: 命中 AGENT_HINTS -> 'agent'; 命中 WORKFLOW_HINTS -> 'workflow';
    #       (若都命中, 以 agent 为先, 因为只要有开放性就值得用 agent)
    #       都没有 -> 'workflow'(默认更可控)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert should_use_agent('每次都按固定模板生成周报') == 'workflow'
assert should_use_agent('研究这个 bug 的根因, 不确定要查哪些文件') == 'agent'
assert should_use_agent('把这批图片统一压缩') == 'workflow'   # 批量
assert should_use_agent('帮我探索一下这个数据集有什么规律') == 'agent'
assert should_use_agent('翻译这句话') == 'workflow'           # 无信号->默认
print('✅ 练习 3 通过：能据任务开放性初步判断该用 agent 还是 workflow')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def summarize_run(result):
    s = result['status']; cost = result.get('cost', 0)
    if s == 'done':
        return f'✓ 完成({result.get("steps","?")}步, ${cost:.4f}): {result.get("answer","")}'
    if s == 'budget_exceeded':
        return f'✗ 超预算(${cost:.4f})'
    if s == 'loop_detected':
        return '✗ 检测到死循环'
    if s == 'max_steps':
        return '✗ 超过最大步数'
    return f'? 未知状态 {s}'

In [ ]:
# 练习 2 参考答案
def make_looping_agent(reg, loop_threshold=3):
    script = [{'stop_reason':'tool_use',
               'tool_calls':[{'id':'x','name':'calculator','input':{'expr':'1+1'}}]}] * 20
    return Agent(MockLLM(script), reg, max_steps=20, loop_threshold=loop_threshold)

In [ ]:
# 练习 3 参考答案
def should_use_agent(task):
    WORKFLOW_HINTS = ['固定', '每次都', '按顺序', '批量', '总是']
    AGENT_HINTS = ['研究', '探索', '根据情况', '不确定', '自己决定']
    if any(h in task for h in AGENT_HINTS):
        return 'agent'
    if any(h in task for h in WORKFLOW_HINTS):
        return 'workflow'
    return 'workflow'

---
## 🧪 真实数据胶囊：把你造的 agent 接到真实 Claude, 端到端跑一个任务

本课的终点：用 `make_llm()` 把你亲手拼的 `Agent` 接到**真实 `claude-opus-4-8`**——有 key 时, 真实模型会在你造的 harness 里**自主**多步完成任务; 无 key 时回退 MockLLM、用同一份 Agent 代码跑通。**这是『同一份代码、两种读者』的最终兑现, 也是整门课的句号。**

> 真实路径需要 `to_api_messages`/`from_api_response`(模块03)做格式翻译; 这里给出完整 `AnthropicLLM` 与 `make_llm`。**无 key 也能跑。**

In [ ]:
# 模块03 的翻译两半(完整版)
def to_api_messages(history):
    out = []
    for m in history:
        if m['role'] == 'assistant':
            blocks = []
            if m.get('text'): blocks.append({'type':'text','text':m['text']})
            for call in (m.get('tool_calls') or []):
                blocks.append({'type':'tool_use','id':call['id'],'name':call['name'],'input':call['input']})
            out.append({'role':'assistant','content':blocks or m.get('text','')})
        else:
            content = m['content']
            if isinstance(content, list):
                out.append({'role':'user','content':[
                    {'type':'tool_result','tool_use_id':r['tool_use_id'],
                     'content':r['content'],'is_error':r.get('is_error',False)} for r in content]})
            else:
                out.append({'role':'user','content':content})
    return out

def from_api_response(resp):
    text, calls = '', []
    for blk in resp.content:
        if blk.type == 'text': text += blk.text
        elif blk.type == 'tool_use': calls.append({'id':blk.id,'name':blk.name,'input':blk.input})
    return {'stop_reason':resp.stop_reason,'text':text,'tool_calls':calls,
            'usage':{'input_tokens':resp.usage.input_tokens,'output_tokens':resp.usage.output_tokens}}

class AnthropicLLM:
    def __init__(self, model='claude-opus-4-8'):
        import anthropic
        self.client = anthropic.Anthropic(); self.model = model
    def complete(self, system, messages, tools):
        resp = self.client.messages.create(
            model=self.model, max_tokens=1024, system=system or 'You are a helpful agent.',
            messages=to_api_messages(messages), tools=tools or [])
        return from_api_response(resp)

def make_llm(mock_script, model='claude-opus-4-8'):
    if os.environ.get('ANTHROPIC_API_KEY'):
        try:
            import anthropic  # noqa
            return AnthropicLLM(model)
        except ImportError:
            pass
    return MockLLM(mock_script)
print('✅ AnthropicLLM + make_llm 就绪(模块03 完整翻译)')

In [ ]:
# 端到端: 同一份 Agent 代码, 有 key 接真 Claude、没 key 回退 mock
reg_final = ToolRegistry()
reg_final.add('count_letters', lambda word: len(word),
              {'type':'object','properties':{'word':{'type':'string','description':'单词'}},'required':['word']},
              '数一个英文单词的字母数')
reg_final.add('calculator', lambda expr: round(eval(expr, {'__builtins__':{}}, {}), 2),
              {'type':'object','properties':{'expr':{'type':'string','description':'算术表达式'}},'required':['expr']},
              '计算算术表达式')

# 无 key 时这段 mock 脚本驱动; 有 key 时被忽略, 真实 Claude 自主决策
mock_script = [
    {'stop_reason':'tool_use','tool_calls':[{'id':'1','name':'count_letters','input':{'word':'agent'}}]},
    {'stop_reason':'tool_use','tool_calls':[{'id':'2','name':'calculator','input':{'expr':'3.14159*5*5'}}]},
    {'stop_reason':'end_turn','text':'该圆面积约为 78.54'},
]
llm = make_llm(mock_script)
agent = Agent(llm, reg_final, system='你是研究助手, 一步步用工具解决问题。', max_steps=8, max_cost=0.5)
out = agent.run('某圆半径等于 "agent" 这个词的字母数, 求它的面积(用工具)')
print('LLM:', type(llm).__name__)
print('\n'.join(out['trace']))
print('\nstatus:', out['status'], '| cost: $%.4f' % out['cost'])
assert out['status'] in ('done','max_steps','budget_exceeded')
if not os.environ.get('ANTHROPIC_API_KEY'):
    assert out['status'] == 'done' and '78.54' in out['answer']
print('\n✅ 终极胶囊跑通: 你亲手造的 agent 端到端完成多步任务(有 key 接真 Claude、没 key 回退 mock)')

**🧪 胶囊练习**：实现 `run_task_robustly(task, reg, mock_script, max_cost=0.5)`：用 `make_llm` + `Agent` 跑一个任务，**无论真实调用是否出问题都返回一个结果 dict**(真实路径异常时, 用 `summarize_run` 包一个失败摘要回来)。这是整门课的收束: **一个你完全掌控、不会崩、能接真实 Claude 的 agent 入口**。

In [ ]:
def run_task_robustly(task, reg, mock_script, max_cost=0.5):
    # TODO: llm = make_llm(mock_script)
    #       agent = Agent(llm, reg, max_cost=max_cost)
    #       try: out = agent.run(task); 返回 {'result':out, 'summary':summarize_run(out)}
    #       except Exception as e: 返回 {'result':{'status':'error'}, 'summary':f'✗ 运行出错: {e}'}
    raise NotImplementedError

In [ ]:
# 自测: 无论如何都返回结果、不崩
r = run_task_robustly('某圆半径="agent"字母数, 求面积', reg_final, mock_script)
assert 'result' in r and 'summary' in r
assert isinstance(r['summary'], str) and len(r['summary']) > 0
if not os.environ.get('ANTHROPIC_API_KEY'):
    assert r['result']['status'] == 'done'
    assert '✓ 完成' in r['summary']
print('摘要:', r['summary'])
print('✅ 胶囊练习通过: 一个完全掌控、不会崩、能接真实 Claude 的 agent 入口 —— 全课收束!')

In [ ]:
# 📖 胶囊参考答案
def run_task_robustly(task, reg, mock_script, max_cost=0.5):
    try:
        llm = make_llm(mock_script)
        agent = Agent(llm, reg, max_cost=max_cost)
        out = agent.run(task)
        return {'result':out, 'summary':summarize_run(out)}
    except Exception as e:
        return {'result':{'status':'error'}, 'summary':f'✗ 运行出错: {type(e).__name__}: {e}'}

### 小结（全课收束）
- **最小 agent = 循环(01) + 工具(02) 跑在适配器(03)上、外裹鲁棒层(04)**，靠依赖注入拼起来——这一个 `run()` 方法里整门课收束。
- **组装拓扑**：鲁棒层(外) → 循环(中) → {适配器, 工具系统}(被调用)；每块零件接口清晰, 拼装就该平凡。
- **端到端**：自主串起多个工具完成多步任务、中间结果正确回灌; 逆境(幻觉工具)能自我恢复。
- **可配置 + 可扩展**：换大脑/工具/上限/人格不改核心; 在 registry 等接缝处加能力很平滑——好分解让 agent 能长大。
- **agent vs workflow**：不是所有任务都该用 agent; 先判断再动手。
- **你造了一个亲手实现的、彻底理解的、能接真实 Claude 的 agent harness** —— 你不再是框架黑箱的使用者。

🎉 **恭喜完成全课!** 下一步: 设好 `ANTHROPIC_API_KEY`、给 agent 配几个真实工具, 跑一个你自己关心的任务——亲眼看到你造的 harness 驱动真实 Claude 工作。